# ДЗ 6. Очистка orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw06` репозитория `dsa-homework` (`hw06.ipynb`): `git add hw06` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: всё с уроков 1–6. `groupby`, `pivot_table`, `merge` пока не используем (урок 7). Небольшой цикл по городам или месяцам допустим. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
# 2
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
# 3
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))
# 4
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())
# 5
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())
```

Ваши ответы и объяснения:

1. isna.sum() isna turns true where value is missin, sum counts True as 1, so you get 1. There is one missing value. fillna filss none with 0, then 4/3=1.333
2. strip works with spaces, lower lowers and nuninque returns number of distinct values, therefore 2
3. drops second row, len returns number of rows, but initial df was not changed therefore 3
4. changes str to number, then adds them 1005
5. [9, 10] takes only monthes then creates a list from them

In [5]:
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())

[9, 10]


In [4]:
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())

1005


In [3]:
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))

3


In [1]:
import pandas as pd
# 1
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())

1 1.3333333333333333


In [14]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders (3).csv"))

orders.csv на месте: True


## [База] 1. Функция очистки
Напишите функцию `clean(path)`, которая читает orders.csv и делает все шаги урока: суммы в числа, города через словарь, дубли, даты, без TEST. Отменённые заказы **не** удаляйте: они нужны для доли отмен.
После каждого шага функция печатает, сколько строк осталось. Проверки внутри функции: у `amount` нет NaN, у `city` нет NaN, `order_id` уникален.
Сколько строк в результате?

In [19]:
import pandas as pd

def clean(path):
    df = pd.read_csv(path)
    print("Загружено:", len(df))

    # 1. Суммы в числа: "15 990" → 15990
    df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""), errors="coerce")
    print("После сумм:", len(df))

    # 2. Города через словарь
    city_map = {
        "алматы": "Алматы", "almaty": "Алматы",
        "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент",
        "караганда": "Караганда",
        "атырау": "Атырау",
        "актобе": "Актобе",
    }
    df["city"] = df["city"].str.strip().str.lower().map(city_map)
    print("После городов:", len(df))
    df["city"] = df["city"].str.strip().str.lower().map(city_map)
    print("После городов:", len(df))

    # 3. Дубли
    df = df.drop_duplicates()
    print("После удаления дублей:", len(df))

    # 4. Даты
    df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
    print("После дат:", len(df))

    # 5. Без TEST (отменённые НЕ удаляем)
    df = df[df["customer_id"] != "TEST"]
    print("После удаления TEST:", len(df))

    # Проверки
    print("NaN в amount:", df["amount"].isna().sum())
    print("NaN в city:", df["city"].isna().sum())
    print("order_id уникален:", len(df) == df["order_id"].nunique())

    return df


df_clean = clean("orders.csv")
print("Итого строк:", len(df_clean))

Загружено: 5463
После сумм: 5463
После городов: 5463
После городов: 5463
После удаления дублей: 5283
После дат: 5283
После удаления TEST: 5280
NaN в amount: 0
NaN в city: 0
order_id уникален: True
Итого строк: 5280


## [База] 2. Выручка по чистым данным
Выручка оплаченных заказов за сентябрь и октябрь и изменение в %. Затем то же для каждого из 6 городов (цикл по городам).
Совпадает ли с таблицей «город × месяц» урока 4? Какой город падает сильнее всех?

In [23]:
paid = df_clean[df_clean["status"] == "оплачен"]

sep = paid[paid["order_date"] < "2026-10-01"]
octo = paid[paid["order_date"] >= "2026-10-01"]

# Всего
sep_total = sep["amount"].sum()
oct_total = octo["amount"].sum()
change = (oct_total - sep_total) / sep_total * 100
print(f"Всего: сентябрь {sep_total:.0f}, октябрь {oct_total:.0f}, изменение {change:.1f}%")

# По городам
for city in df_clean["city"].unique():
    s = sep[sep["city"] == city]["amount"].sum()
    o = octo[octo["city"] == city]["amount"].sum()
    ch = (o - s)/ s * 100
    print(f"{city}: сентябрь {s:.0f}, октябрь {o:.0f}, изменение {ch:.1f}%")

Всего: сентябрь 35243410, октябрь 34419520, изменение -2.3%
Алматы: сентябрь 14501070, октябрь 14643020, изменение 1.0%
Астана: сентябрь 9034220, октябрь 8850530, изменение -2.0%
Караганда: сентябрь 3648520, октябрь 3371680, изменение -7.6%
Шымкент: сентябрь 4558650, октябрь 4044010, изменение -11.3%
Атырау: сентябрь 923990, октябрь 969860, изменение 5.0%
Актобе: сентябрь 2576960, октябрь 2540420, изменение -1.4%


In [24]:
paid = df_clean[df_clean["status"] == "оплачен"]
paid

,order_id,order_date,customer_id,city,category,amount,delivery_days,status
0,101674,2026-09-01 08:00:00,C00681,Алматы,Одежда,8070.0,1.0,оплачен
1,101214,2026-09-01 08:00:00,C01210,Астана,Красота,3630.0,2.0,оплачен
2,102424,2026-09-01 08:00:00,C01408,Караганда,Дом,9120.0,3.0,оплачен
3,102333,2026-09-01 08:00:00,C01529,Алматы,Одежда,9350.0,2.0,оплачен
4,102564,2026-09-01 09:00:00,C00657,Астана,Дом,11010.0,1.0,оплачен
...,...,...,...,...,...,...,...,...
5457,103557,2026-10-31 21:00:00,C01573,Алматы,Дом,9170.0,2.0,оплачен
5458,104168,2026-10-31 21:00:00,C01057,Алматы,Одежда,11430.0,2.0,оплачен
5459,104648,2026-10-31 21:00:00,C00629,Караганда,Книги,3640.0,2.0,оплачен
5460,104615,2026-10-31 22:00:00,C00557,Астана,Электроника,22150.0,NaN,оплачен


## [База] 3. Журнал очистки для менеджера
Markdown-таблица: шаг очистки → сколько строк убрал → как изменилось «падение выручки» (от −22,7% до −2,3%).
Одной фразой для менеджера: какой шаг сильнее всего исказил его цифру?

In [26]:
city_map = {
    "алматы": "Алматы", "almaty": "Алматы",
    "астана": "Астана", "astana": "Астана",
    "шымкент": "Шымкент", "shymkent": "Шымкент",
    "караганда": "Караганда",
    "атырау": "Атырау",
    "актобе": "Актобе",
}

In [27]:
def revenue_drop(d):
    p = d[d["status"] == "оплачен"]
    s = p[p["order_date"] < "2026-10-01"]["amount"].sum()
    o = p[p["order_date"] >= "2026-10-01"]["amount"].sum()
    return (o - s) / s * 100

d = pd.read_csv("orders.csv")

# 1. Суммы в числа
before = len(d)
d["amount"] = pd.to_numeric(d["amount"].str.replace(" ", ""), errors="coerce")
print(f"Суммы:   убрано {before - len(d)}, падение {revenue_drop(d):.1f}%")

# 2. Города
before = len(d)
d["city"] = d["city"].str.strip().str.lower().map(city_map)
print(f"Города:  убрано {before - len(d)}, падение {revenue_drop(d):.1f}%")

# 3. Дубли
before = len(d)
d = d.drop_duplicates()
print(f"Дубли:   убрано {before - len(d)}, падение {revenue_drop(d):.1f}%")

# 4. Даты
before = len(d)
d["order_date"] = pd.to_datetime(d["order_date"], errors="coerce")
print(f"Даты:    убрано {before - len(d)}, падение {revenue_drop(d):.1f}%")

# 5. TEST
before = len(d)
d = d[d["customer_id"] != "TEST"]
print(f"TEST:    убрано {before - len(d)}, падение {revenue_drop(d):.1f}%")

Суммы:   убрано 0, падение -22.7%
Города:  убрано 0, падение -22.7%
Дубли:   убрано 180, падение -19.6%
Даты:    убрано 0, падение -19.6%
TEST:    убрано 3, падение -2.3%


## [Стандарт] 4. Пропуски в сроках доставки
Сколько пропусков в `delivery_days` по чистым данным? В каких городах и статусах их больше?
Доля доставки за 2 дня в Шымкенте: (а) `dropna`, (б) `fillna(медиана)`, (в) `fillna(0)`. Какой вариант честный и почему другие вводят в заблуждение?

In [ ]:
# 4

## [Стандарт] 5. Исправьте код от AI
AI написал «очистку» и получил 10 городов вместо 6, а строк осталось 5 463. В коде **две** ошибки. Найдите их и исправьте.
```python
df = pd.read_csv("orders.csv")
df.drop_duplicates()
df["city"] = df["city"].str.lower()
print(df["city"].nunique(), len(df))
```

In [ ]:
# 5

## [Стандарт] 6. Дни недели
По чистым данным: сколько заказов в каждый день недели (`.dt.day_name()` и `value_counts`)? В какой день заказов меньше всего и на сколько процентов меньше, чем в самый загруженный?
Что это значит для графика работы курьеров?

In [ ]:
# 6

## [Челлендж] 7. Когда сломалась доставка в Шымкенте
Средний срок доставки в Шымкенте в сентябре и в октябре. Найдите первую дату, с которой срок доставки в Шымкенте не бывает меньше 5 дней (фильтр + `sort_values` по дате).
Проверьте, что в других городах в эту дату ничего не изменилось. Напишите менеджеру: что случилось, когда и что проверить в первую очередь.

In [ ]:
# 7

## [Челлендж] 8. Надёжный перевод сумм
Напишите функцию `to_amount(s)` для Series строк, которая понимает `"15 990"`, `"15990.0"`, `"1 200,50"` (запятая как разделитель дробной части) и превращает всё остальное (`"нет данных"`) в NaN.
Проверьте на `pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])` и на столбце `amount` из orders.csv: сумма должна совпасть с уроком.

In [ ]:
# 8